In [1]:
# 01_preprocessing.ipynb
# Load CSVs → Merge → Filter failures → Build "document" text
# Output: outputs/final_failed_transactions.csv

import pandas as pd

# Load all CSVs
banks = pd.read_csv("../data/banks.csv")
merchants = pd.read_csv("../data/merchants.csv")
users = pd.read_csv("../data/users.csv")
tx = pd.read_csv("../data/transactions.csv")

In [2]:
banks.head()

,bank_id,bank_name,server_status
0,1,ICICI,DOWN
1,2,HDFC,DOWN
2,3,SBI,DOWN
3,4,Axis,ONLINE
4,5,Kotak,DELAYED


In [3]:
merchants.head()

,merchant_id,name,category
0,1,"Mcgrath, Graham and Jensen",Travel
1,2,Bryant-Tate,Entertainment
2,3,Wall LLC,Food
3,4,Kelly PLC,Entertainment
4,5,Hayes and Sons,Ecommerce


In [4]:
users.head()

,user_id,name,city
0,1,Kimberly Kennedy,Wilsonport
1,2,Ashley Espinoza,Lammouth
2,3,David Holloway,North Denise
3,4,Michelle Parker,Davidshire
4,5,Joshua Richardson,West Eric


In [5]:
tx.head()

,txn_id,user_id,merchant_id,bank_id,txn_time,amount,status,failure_reason
0,de3bfd39-52ad-4ac9-a967-895000358084,787,112,9,21-03-2025 17:17,4859.57,SUCCESS,NaN
1,bfa5412a-a12f-4b43-b448-0a1230e995f8,242,68,1,29-10-2024 17:10,6541.02,SUCCESS,NaN
2,ac0da711-f36e-46a6-8335-22f9e3f6380a,22,18,10,08-03-2025 06:46,5127.52,SUCCESS,NaN
3,a8936e8b-c631-4881-8467-2b4eb0f35c1d,203,32,5,23-12-2024 05:54,4731.21,SUCCESS,NaN
4,c2ed665a-52be-4474-aef0-2962d438f6e1,99,76,10,26-10-2024 04:52,5147.39,SUCCESS,NaN


In [6]:
# Filter failed transactions
tx["txn_time"] = pd.to_datetime(tx["txn_time"], format="%d-%m-%Y %H:%M")
failed = tx[tx["status"] != "SUCCESS"].copy()

# Merge everything
merged = (
    failed
    .merge(banks, on="bank_id", how="left")
    .merge(merchants, on="merchant_id", how="left")
    .merge(users, on="user_id", how="left")
)

# Fix column names
merged = merged.rename(columns={
    "name_x": "merchant_name",
    "name_y": "user_name",
    "city": "user_city"
})

In [9]:
merged.isna().sum()

txn_id            0
user_id           0
merchant_id       0
bank_id           0
txn_time          0
amount            0
status            0
failure_reason    0
bank_name         0
server_status     0
merchant_name     0
category          0
user_name         0
user_city         0
dtype: int64

In [10]:
# Build RAG-ready document
def make_doc(r):
    return f"""
Transaction ID: {r['txn_id']}
Timestamp: {r['txn_time']}
Amount: {r['amount']}
Status: {r['status']}
Failure Reason: {r['failure_reason']}
Bank: {r['bank_name']} (Server: {r['server_status']})
Merchant: {r['merchant_name']} ({r['category']})
User City: {r['user_city']}
"""

merged["document"] = merged.apply(make_doc, axis=1)

# Remove NA failure_reason
merged = merged.dropna(subset=["failure_reason"]).reset_index(drop=True)

In [12]:
# Save
merged.to_csv("../outputs/final_failed_transactions.csv", index=False)
merged.head()

,txn_id,user_id,merchant_id,bank_id,txn_time,amount,status,failure_reason,bank_name,server_status,merchant_name,category,user_name,user_city,document
0,30e2c03e-a070-4076-afa6-e7e7c0448834,897,120,17,2025-01-01 14:59:00,1945.26,FAILED,CARD_DECLINED,Standard Chartered,DOWN,Roach-Edwards,Grocery,Rose Foster,Normanhaven,\nTransaction ID: 30e2c03e-a070-4076-afa6-e7e7...
1,06a16bff-daca-4322-8209-9b2f5d4ddc0a,409,195,5,2025-06-17 17:01:00,5958.95,FAILED,CARD_DECLINED,Kotak,DELAYED,"Armstrong, Adams and Dunlap",Grocery,Margaret Bates,Port Ricardo,\nTransaction ID: 06a16bff-daca-4322-8209-9b2f...
2,cfe8af4a-6bf2-482d-8f49-91af62f352af,825,114,12,2024-11-09 21:20:00,9051.04,FAILED,INVALID_PIN,IDFC,ONLINE,Collins-Evans,Travel,Amber Cooper,Kramerville,\nTransaction ID: cfe8af4a-6bf2-482d-8f49-91af...
3,9d791a99-1ccd-4f15-95fd-14668d8e70c9,462,86,17,2024-07-21 17:54:00,8163.24,FAILED,TIMEOUT,Standard Chartered,DOWN,Myers Group,Ecommerce,Kelly Bruce,West Cindy,\nTransaction ID: 9d791a99-1ccd-4f15-95fd-1466...
4,ab3104f8-8187-44cf-bd19-a78f6d374871,22,66,12,2024-07-13 02:22:00,535.25,FAILED,CARD_DECLINED,IDFC,ONLINE,Berg-Ramos,Fashion,Vincent Lynn,Triciamouth,\nTransaction ID: ab3104f8-8187-44cf-bd19-a78f...
